# Build training-ticket embeddings

This notebook does five simple things:

1. Installs the libraries needed in Colab.
2. Downloads the Bitext customer-support CSV.
3. Uses the exact split function from `pipeline/ingest.py` to recreate the seed-42, intent-stratified 80/20 split.
4. Keeps only training rows and embeds each `instruction` with `BAAI/bge-small-en-v1.5` on the GPU. The vectors are normalized and have 384 dimensions.
5. Saves the requested columns to `embeddings.parquet` and downloads the file.

Embedding only the training rows prevents test-ticket text from leaking into later retrieval or evaluation. Before running, choose **Runtime → Change runtime type → GPU**.

In [ ]:
!pip -q install sentence-transformers pyarrow

In [ ]:
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import torch
from sentence_transformers import SentenceTransformer

DATA_URL = (
    "https://huggingface.co/datasets/bitext/"
    "Bitext-customer-support-llm-chatbot-training-dataset/resolve/main/"
    "Bitext_Sample_Customer_Support_Training_Dataset_27K_responses-v11.csv"
)
CSV_PATH = Path("Bitext_Sample_Customer_Support_Training_Dataset_27K_responses-v11.csv")
urllib.request.urlretrieve(DATA_URL, CSV_PATH)
print(f"Downloaded {CSV_PATH}")

In [ ]:
# Copied verbatim from pipeline/ingest.py.
def stratified_split(
    df: pd.DataFrame,
    stratify_col: str = "intent",
    test_size: float = 0.2,
    seed: int = 42,
) -> pd.DataFrame:
    """Deterministically split dataframe into train and test sets stratified by stratify_col."""
    shuffled = df.sample(frac=1.0, random_state=seed)
    group_sizes = shuffled.groupby(stratify_col)[stratify_col].transform("count")
    rank = shuffled.groupby(stratify_col).cumcount()
    n_train = (group_sizes * (1.0 - test_size)).round().astype(int)
    shuffled["split"] = np.where(rank < n_train, "train", "test")
    return shuffled.sort_index()

df = pd.read_csv(CSV_PATH)
df = stratified_split(df, stratify_col="intent", test_size=0.2, seed=42)
train_df = df.loc[
    df["split"].eq("train"),
    ["instruction", "intent", "category"],
].reset_index(drop=True)

print(df["split"].value_counts().to_string())
print(f"Training rows to embed: {len(train_df)}")

In [ ]:
assert torch.cuda.is_available(), "Select a GPU runtime in Colab before embedding."

model = SentenceTransformer("BAAI/bge-small-en-v1.5", device="cuda")
embeddings = model.encode(
    train_df["instruction"].tolist(),
    batch_size=256,
    normalize_embeddings=True,
    show_progress_bar=True,
    convert_to_numpy=True,
)
assert embeddings.shape == (len(train_df), 384)
print(f"Created embeddings with shape {embeddings.shape}")

In [ ]:
output = pa.table({
    "instruction": pa.array(train_df["instruction"], type=pa.string()),
    "intent": pa.array(train_df["intent"], type=pa.string()),
    "category": pa.array(train_df["category"], type=pa.string()),
    "embedding": pa.array(
        embeddings.astype(np.float32).tolist(),
        type=pa.list_(pa.float32()),
    ),
})
pq.write_table(output, "embeddings.parquet")
print(f"Saved {output.num_rows} rows to embeddings.parquet")

In [ ]:
from google.colab import files

files.download("embeddings.parquet")